<a href="https://colab.research.google.com/github/KCL-Health-NLP/nlp_examples/blob/master/ann/rag_langchain_huggingface_pubmed_hosted_runtime.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install packages.

!pip install --upgrade --quiet transformers
!pip install --upgrade --quiet langchain langchain_community
!pip install --upgrade --quiet langchain-huggingface
!pip install --quiet huggingface_hub<=0.27.1
!pip install rank_bm25

In [ ]:
# Import packages.
import pandas as pd
import re, json
from langchain_community.document_loaders import DataFrameLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.retrievers import BM25Retriever
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

import langchain_community
from langchain_huggingface import HuggingFaceEndpoint
from langchain.chains import LLMChain
import torch

In [ ]:
# 1. Download sample pubmed data from: https://drive.google.com/file/d/18rY-9JoZM6epbhwvTxahxMSUkonEkPtB/view?usp=drive_link

# 2. Move drag and drop into folder icon on the left hand side of colab (🗀 <-----)


# You need to get your access token from huggingface, run this cell and paste
# it in to the resulting prompt, for use in later sections
# How to get a token is described here:
# https://huggingface.co/docs/api-inference/quicktour#get-your-api-token

# getpass provides an obscured password prompt.
# os gives access to operating ssytem functionality, which
# we need to set an environment-wide variable to hold our token
from getpass import getpass
import os

HUGGINGFACEHUB_API_TOKEN = getpass("Paste in your API token and press enter")

# We put the token in an environment variable, from where LangChain will access it when needed
os.environ["HUGGINGFACEHUB_API_TOKEN"] = HUGGINGFACEHUB_API_TOKEN

# Load PubMed data filtered for 10,000 entries containing the following keywords.



In [ ]:
df = pd.read_csv('/content/pubmed_samples.csv')
# 10000 Pubmed abstracts containing:
# 'aspirin',
#  'coumadin',
#  'warfarin',
#  'docusate sodium',
#  'methadone',
#  'tizanidine',
#  'senna',
#  'omeprazole',
#  'simvastatin',
#  'nitroglycerin',
#  'chest pain',
#  'fever',
#  'shortness of breath',
#  'sob',
#  'orthopnea',
#  'dysuria',
#  'headache',
#  'constipation',
#  'diarrhea',
#  'cough'
df['Authors'] = df['Authors'].astype(str)

In [ ]:
df

RAG search for PubMed data

In [ ]:
# Transform data into Documents, where Abstract are main text content to search
loader = DataFrameLoader(df, page_content_column="Abstract")
docs = loader.load()

In [ ]:
# Use this cell to explore the dataset using Pandas and think of potential queries for an LLM.
docs[0]

In [ ]:
# backup if endpoint not working

# Make sure runtime has a GPU if not using endpoint.



# Using the paid for model endpoint, which can host a wider range of models
# This is the url of a paid for endpoint - replace with whichever you are using
# You need to enter the URL provided for the practical!
endpoint_url = "https://f9ctlf1rpteiq2sb.eu-west-1.aws.endpoints.huggingface.cloud"

# Make the model - we are starting to use more parameters
llm = HuggingFaceEndpoint(
    endpoint_url=f"{endpoint_url}",
    max_new_tokens=1024,
    temperature=0.1,
)

# Invoke the model directly
llm("A working LLM endpoint would say ")


In [ ]:
# Split the loaded documents into smaller chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

splits = text_splitter.split_documents(docs)

# Create a BM25Retriever from the split documents - this is an extension of TF-IDF
# k=5 specifies that the retriever should return the top 5 most relevant chunks - have to keep this low if running out of tokens.
retriever = BM25Retriever.from_documents(splits, k=5)

In [ ]:
# Define the prompt using Llama's chat tokens for consistent output.
prompt = PromptTemplate.from_template(
    """<|begin_of_text|><|start_header_id|>system<|end_header_id|>You are a helpful assistant for answering questions using *only* the provided context.
Do not use any external or prior knowledge. Only answer the questions asked. If the answer is not apparent from the context, say 'I don't know'.<|eot_id|><|start_header_id|>user<|end_header_id|>

Answer my question based on the following documents:
{context}

Question: {question}<|eot_id|><|start_header_id|>assistant<|end_header_id|>"""
)

# Function to format documents
def format_docs(docs):
    return "\n\n".join(
        f"Content: {doc.page_content}\nMetadata: {doc.metadata}" for doc in docs
    )

# Build RAG chain
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
)

In [ ]:
# Example question using non-RAG pipeline.
question = "Please tell me what COPD is? Then list papers that mention it."
response = llm.invoke(question)
print(response)

In [ ]:
# Example question using RAG pipeline.
question = "Please tell me what COPD is? Then list papers that mention it."
response = rag_chain.invoke(question)

print(response)

In [ ]:
# Using the syntax above explore other questions for the model - how does it do? Does it say when it doesn't know?

question = "List the titles of research papers focused on patients with atrial fibrillation."
response = rag_chain.invoke(question)

print(response)

In [ ]:
# What about out of scope topics?

question = "List the titles of research papers focused on patients with the pokemon Pikachu."
response = rag_chain.invoke(question)

print(response)

In [ ]:
# Play with different queries - bear in mind we are using a very 'small' llm.
# Real world RAG systems are likely to use a much larger generation model with better instruction following.
